In [ ]:
from Datasets.mapping import nationality_to_country
from Datasets.politicians.politicians import Politicians
from Datasets.utilities import get_person_nationality_with_dbpedia_link, get_country, country_name_standardizer

import os
os.environ['NationalityDatasets'] = "./data/"
politicians = Politicians()

In [ ]:
raw_data = politicians.get_raw_data()
raw_data.shape

In [ ]:
raw_data.head()

### Exploring the data


##### Checking None values

In [ ]:
raw_data.fillna('', inplace=True)

In [ ]:
raw_data[raw_data['#DBpURL']=='']

In [ ]:
raw_data['full-name'] = raw_data['#DBpURL'].apply(lambda x: " ".join(x.split('/')[-1].split('_')))

In [ ]:
raw_data['full-name']

In [ ]:
without_nationality = raw_data[raw_data['nationality']=='']
without_nationality

In [ ]:
without_nationality.shape

In [ ]:
raw_data.shape[0] - without_nationality.shape[0]

In [ ]:
raw_data['nationality'] = raw_data['nationality'].apply(lambda x: x.replace("'", '').replace('[','').replace(']','').split(',')[-1].strip())
raw_data

In [ ]:
raw_data['country'] = raw_data.nationality.apply(lambda x: nationality_to_country(x) if x != '' else '')

In [ ]:
raw_data[raw_data.country == ''].nationality.value_counts()

In [ ]:
raw_data[raw_data.country == '']

In [ ]:
raw_data.country = raw_data.apply(lambda x: get_country(get_person_nationality_with_dbpedia_link(x['#DBpURL'])) if x.country == '' else x.country, axis=1)


In [ ]:
raw_data.country = raw_data.country.apply(lambda x: country_name_standardizer(x))

In [ ]:
raw_data = raw_data[raw_data.country != '']

##### Checking duplicate values

In [ ]:
raw_data['full-name'].value_counts()

### Saving data

In [ ]:
raw_data.shape

In [ ]:
raw_data['id'] = raw_data.apply(lambda x:'politicians-'+str(x.name), axis=1)

In [ ]:
data = raw_data[['id', 'full-name', 'country']]
data.set_index('id', inplace=True)

In [ ]:
data.shape

In [ ]:
data.country.unique()

In [ ]:
data.country.value_counts()

In [ ]:
data.country.value_counts()[:30].plot.bar()

In [ ]:
politicians.save_data(data)

In [ ]:
politicians.data = data

In [ ]:
politicians.load_data()
politicians.save_split_data()